# M7 Validation: Real RGB Perception + LLM Plan + LIBERO-10 Task 0 (Linux/Colab)

## 1 — Secrets & clone

In [ ]:
import getpass, os

github_token = getpass.getpass('GitHub PAT (repo scope): ').strip()
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ').strip()
assert github_token != '', f'github_token was not set'

In [ ]:
import os, subprocess, sys

REPO_DIR = '/content/Longor'
GITHUB_REPO = os.environ.get('GITHUB_REPO', 'pmish11/Longor-Robotics')

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Existing clone detected -> git pull')
    subprocess.run(['git', 'pull', '--ff-only'], check=True, cwd=REPO_DIR)
else:
    clone_url = f"https://{github_token}@github.com/{GITHUB_REPO}.git"
    subprocess.run(['git', 'clone', clone_url, REPO_DIR], check=True)
    print('Cloned OK')

os.chdir(REPO_DIR)

## 2 — App deps + LIBERO + headless

In [ ]:
import importlib.util
import logging
import os
import subprocess
import sys

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

# Colab images can have stale apt indexes; refresh first to avoid mesa package 404s.
!apt-get update -y
!apt-get install -y --fix-missing cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r requirements-libero.txt

spec = importlib.util.find_spec('robosuite')
if spec:
    script = spec.submodule_search_locations[0] + '/scripts/setup_macros.py'
    result = subprocess.run([sys.executable, script], check=False)
    if result.returncode != 0:
        logging.warning('robosuite setup_macros.py failed (rc=%s); continuing', result.returncode)

## 3 — SmolVLA VLA venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-smolvla-linux.txt')
print('SmolVLA VLA venv ready.')

## 4 — Env config (M7: PERCEPTION_BACKEND=real)

In [ ]:
import importlib.util, os, yaml

env_content = f"""OPENAI_API_KEY={os.environ['OPENAI_API_KEY']}
PERCEPTION_BACKEND=real
LLM_MODEL=gpt-4.1-mini
LOG_LEVEL=DEBUG
VLA_CHUNK_STEPS=10
VLA_STEPS_PER_ACTION=1
VLA_ACTION_SCALE=1.0
LIBERO_EE_DELTA_SCALE=1.0
"""
with open('/content/Longor/.env', 'w') as f:
    f.write(env_content)
print('Wrote /content/Longor/.env (PERCEPTION_BACKEND=real, VLA_CHUNK_STEPS=10)')

libero_config_path = os.environ.get('LIBERO_CONFIG_PATH', os.path.expanduser('~/.libero'))
config_file = os.path.join(libero_config_path, 'config.yaml')

if not os.path.exists(config_file):
    os.makedirs(libero_config_path, exist_ok=True)
    spec = importlib.util.find_spec('libero')
    libero_pkg = os.path.join(os.path.dirname(spec.origin), 'libero')
    default_config = {
        'benchmark_root': libero_pkg,
        'bddl_files': os.path.join(libero_pkg, 'bddl_files'),
        'init_states': os.path.join(libero_pkg, 'init_files'),
        'datasets': os.path.join(os.path.dirname(libero_pkg), 'datasets'),
        'assets': os.path.join(libero_pkg, 'assets'),
    }
    with open(config_file, 'w') as f:
        yaml.dump(default_config, f)
    print(f'Created {config_file}')
else:
    print(f'Already exists: {config_file}')

## 5 — Start SmolVLA VLA server + health

In [ ]:
import json, os, subprocess, time, urllib.request

VLA_LOG = '/tmp/m7_vla_smolvla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
POLL_SEC = 15
MAX_WAIT_SEC = int(os.environ.get('VLA_STARTUP_MAX_WAIT_SEC', 120))
MAX_STALL_SEC = int(os.environ.get('VLA_STARTUP_MAX_STALL_SEC', 60))
VLA_STALE_PATTERNS = ('run_vla_server.py', 'smolvla_service.py')


def _read_log_snippet(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'


def _stop_vla(proc, log_handle):
    if log_handle and not log_handle.closed:
        log_handle.flush()
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
    if log_handle and not log_handle.closed:
        log_handle.close()


def _startup_failure(reason, proc, log_handle, start_time):
    elapsed = time.monotonic() - start_time
    snippet = _read_log_snippet(40)
    _stop_vla(proc, log_handle)
    hints = (
        'Likely causes: HF_TOKEN missing/invalid (checkpoint download fails), '
        'very long first-time checkpoint download, or GPU OOM during model load. '
        f'Check HF_TOKEN, disk space, and full log at {VLA_LOG}.'
    )
    raise RuntimeError(
        f'{reason} after {elapsed:.0f}s.\n'
        f'Recent log:\n{snippet}\n'
        f'{hints}'
    )


def _print_new_log(log_handle, log_offset):
    try:
        log_handle.flush()
        with open(VLA_LOG) as f:
            f.seek(log_offset)
            new_chunk = f.read()
            new_offset = f.tell()
        if new_chunk:
            print(new_chunk, end='')
        return new_offset, bool(new_chunk)
    except FileNotFoundError:
        return log_offset, False


def _cleanup_stale_vla_processes():
    port_result = subprocess.run(
        ['fuser', '-k', '8765/tcp'], capture_output=True, text=True,
    )
    port_killed = bool(port_result.stdout.strip() or port_result.stderr.strip())

    stale_pids = []
    for pattern in VLA_STALE_PATTERNS:
        pgrep = subprocess.run(
            ['pgrep', '-f', pattern], capture_output=True, text=True,
        )
        for line in pgrep.stdout.splitlines():
            pid = line.strip()
            if pid.isdigit():
                stale_pids.append(int(pid))

    stale_pids = sorted(set(stale_pids))
    for pid in stale_pids:
        subprocess.run(['kill', '-9', str(pid)], capture_output=True)

    if port_killed or stale_pids:
        print(
            f'Cleanup: port8765={port_killed}, '
            f'stale_pids={stale_pids or "none"}'
        )
    else:
        print('Cleanup: no stale VLA processes found')


_cleanup_stale_vla_processes()
print(
    f'Startup limits: hard={MAX_WAIT_SEC}s, stall={MAX_STALL_SEC}s '
    f'(override via VLA_STARTUP_MAX_WAIT_SEC / VLA_STARTUP_MAX_STALL_SEC)'
)

log = open(VLA_LOG, 'w', buffering=1)
# Server-side action queue batch size (matches libero_smoke apply_smoke_env_defaults).
VLA_CHUNK_STEPS = '10'
env_for_server = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'VLA_POLICY': 'smolvla',
    'SMOLVLA_CHECKPOINT': 'HuggingFaceVLA/smolvla_libero',
    'VLA_CHUNK_STEPS': VLA_CHUNK_STEPS,
}
print(f'Server queue env: VLA_CHUNK_STEPS={VLA_CHUNK_STEPS} (actions per /predict)')
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor',
    env=env_for_server,
)

start_time = time.monotonic()
log_offset = 0
last_log_progress = start_time
health_ok = False

try:
    while time.monotonic() - start_time < MAX_WAIT_SEC:
        time.sleep(POLL_SEC)
        elapsed = time.monotonic() - start_time

        if vla_proc.poll() is not None:
            log_offset, _ = _print_new_log(log, log_offset)
            _startup_failure(
                f'VLA server exited early (code {vla_proc.returncode})',
                vla_proc, log, start_time,
            )

        log_offset, had_new_log = _print_new_log(log, log_offset)
        if had_new_log:
            last_log_progress = time.monotonic()

        try:
            resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
            body = json.loads(resp.read())
            log_offset, _ = _print_new_log(log, log_offset)
            print(f'\nVLA server ready ({elapsed:.0f}s):', body)
            assert body.get('model') == 'smolvla', body
            health_ok = True
            break
        except Exception:
            print(f'Status: waiting for /health ({elapsed:.0f}s)')
            if time.monotonic() - last_log_progress >= MAX_STALL_SEC:
                log_offset, _ = _print_new_log(log, log_offset)
                _startup_failure(
                    f'[stall timeout] VLA server startup stalled '
                    f'(no new log output for {MAX_STALL_SEC}s; '
                    f'global cap {MAX_WAIT_SEC}s)',
                    vla_proc, log, start_time,
                )

    if not health_ok:
        log_offset, _ = _print_new_log(log, log_offset)
        _startup_failure(
            f'[hard timeout] VLA server did not become healthy within '
            f'{MAX_WAIT_SEC}s',
            vla_proc, log, start_time,
        )
except KeyboardInterrupt:
    print('Startup interrupted; stopping VLA server...')
    raise
finally:
    if not health_ok:
        _stop_vla(vla_proc, log)

## 6 — Gold reference (LeRobot eval) vs Longor smoke

Run official LeRobot evaluation for `HuggingFaceVLA/smolvla_libero` on **LIBERO-10 task 0**, then the Longor grasp smoke in the same runtime. Compare logs to see whether failures come from Longor integration or the checkpoint/policy itself.

**Runtime note:** Gold eval runs the policy in-process (not via §5 HTTP server). If VRAM is tight, stop §5 before gold eval, then run §5 again before the Longor smoke cell below (it requires the server).

In [ ]:
import os, shutil, subprocess
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
gold_log = f'{trace_dir}/gold_lerobot_eval_{stamp}.txt'
os.environ['GOLD_EVAL_LOG'] = gold_log

vla_python = '/content/Longor/.vla-venv/bin/python'
vla_bin = '/content/Longor/.vla-venv/bin'
eval_out = f'/content/Longor/traces/gold_lerobot_eval_out_{stamp}'

env = {
    **os.environ,
    'MUJOCO_GL': 'egl',
    'PYOPENGL_PLATFORM': 'egl',
    'HF_HUB_DISABLE_PROGRESS_BARS': '1',
    'TQDM_DISABLE': '1',
}

print(f'Capture: {gold_log}')
logs = []

# Ensure .vla-venv has LIBERO eval extras before invoking official eval CLI.
deps_cmd = [vla_python, '-m', 'pip', 'install', '-q', 'lerobot[libero]>=0.5']
print(f'Deps command: {" ".join(deps_cmd)}')
deps = subprocess.run(
    deps_cmd, cwd='/content/Longor', env=env,
    capture_output=True, text=True, timeout=1200,
)
logs.append(
    '=== deps install ===\n'
    f'command: {" ".join(deps_cmd)}\n'
    f'exit_code: {deps.returncode}\n\n'
    f'--- stdout ---\n{deps.stdout}\n'
    f'--- stderr ---\n{deps.stderr}\n'
)
if deps.returncode != 0:
    with open(gold_log, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(logs))
    raise RuntimeError(f'Failed to prepare .vla-venv eval deps (exit={deps.returncode})')

base_args = [
    '--policy.path=HuggingFaceVLA/smolvla_libero',
    '--policy.n_action_steps=10',
    '--policy.device=cuda',
    '--env.type=libero',
    '--env.task=libero_10',
    '--env.task_ids=[0]',
    '--eval.batch_size=1',
    '--eval.n_episodes=3',
    '--env.max_parallel_tasks=1',
    f'--output_dir={eval_out}',
]

candidates = []
if shutil.which('lerobot-eval', path=vla_bin):
    candidates.append([f'{vla_bin}/lerobot-eval', *base_args])
candidates.append([vla_python, '-m', 'lerobot.scripts.lerobot_eval', *base_args])

last_error = None
for cmd in candidates:
    print(f'\n=== Trying: {" ".join(cmd)} ===')
    result = subprocess.run(
        cmd, cwd='/content/Longor', env=env,
        capture_output=True, text=True, timeout=1800,
    )
    logs.append(
        f'command: {" ".join(cmd)}\n'
        f'exit_code: {result.returncode}\n\n'
        f'--- stdout ---\n{result.stdout}\n'
        f'--- stderr ---\n{result.stderr}\n'
    )
    with open(gold_log, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(logs))
    print(f'Command: {" ".join(cmd)}')
    print(f'Exit code: {result.returncode}')
    if result.returncode == 0:
        print('Gold LeRobot eval command completed.')
        break
    snippet = (result.stderr or result.stdout)[-800:]
    last_error = snippet
    print('Failed; snippet:\n', snippet)
else:
    print('All LeRobot eval variants failed. See log for details.')
    if last_error:
        print('Last error snippet:\n', last_error)

In [ ]:
import os, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
longor_log = f'{trace_dir}/longor_grasp_smoke_{stamp}.txt'
os.environ['LONGOR_GRASP_LOG'] = longor_log

# Match §8 grasp-debug telemetry; server from §5 must still be up.
os.environ.setdefault('LIBERO_VLA_STATE_MODE', 'eef_axis_angle')
os.environ['LIBERO_VLA_STATE_DEBUG'] = '1'
os.environ['VLA_DEBUG_ACTION_TELEMETRY'] = '1'

cmd = [
    sys.executable, '-m', 'pytest',
    '-m', 'integration',
    'tests/test_libero_integration.py::test_grasp_smoke',
    '-v', '-s',
]
env = {
    **os.environ,
    'PYTHONPATH': '/content/Longor/src',
    'MUJOCO_GL': 'egl',
    'PYOPENGL_PLATFORM': 'egl',
}

print(f'Capture: {longor_log}')
print(f'Command: {" ".join(cmd)}')
result = subprocess.run(
    cmd, cwd='/content/Longor', env=env,
    capture_output=True, text=True, timeout=900,
)
combined = (
    f'command: {" ".join(cmd)}\n'
    f'exit_code: {result.returncode}\n\n'
    f'--- stdout ---\n{result.stdout}\n'
    f'--- stderr ---\n{result.stderr}\n'
)
with open(longor_log, 'w', encoding='utf-8') as f:
    f.write(combined)
print(f'Exit code: {result.returncode}')
print(combined[-4000:])

In [ ]:
import glob
import os
import re

trace_dir = '/content/Longor/traces'

def _latest(pattern: str, env_key: str) -> str | None:
    explicit = os.environ.get(env_key, '').strip()
    if explicit and os.path.exists(explicit):
        return explicit
    matches = sorted(glob.glob(f'{trace_dir}/{pattern}'))
    return matches[-1] if matches else None

gold_log = _latest('gold_lerobot_eval_*.txt', 'GOLD_EVAL_LOG')
longor_log = _latest('longor_grasp_smoke_*.txt', 'LONGOR_GRASP_LOG')

PATTERNS = {
    'exit_code': re.compile(r'exit_code:\s*(-?\d+)', re.I),
    'success_rate': re.compile(r'success(?:_rate)?\s*[=:]\s*([0-9.]+)\s*%?', re.I),
    'success_count': re.compile(r'(?:success(?:_count|es)?|solved)\s*[=:]?\s*(\d+)\s*/\s*(\d+)', re.I),
    'episode_success': re.compile(r'episode.*?(?:success|done).*?(true|1|passed)', re.I),
    'episode_fail': re.compile(r'episode.*?(?:fail|error).*?(true|1)', re.I),
    'min_dist': re.compile(r'min_dist\s*[=:]\s*([0-9.]+)', re.I),
    'grasp': re.compile(r'grasp[_ ]?(?:confirmed|triggered|lift)?\s*[=:]\s*(\w+)', re.I),
    'raw_mag': re.compile(r'raw_mag\s*[=:]\s*([0-9.]+)', re.I),
    'action_mag': re.compile(r'(?:action|delta)[-_ ]?mag(?:nitude)?\s*[=:]\s*([0-9.]+)', re.I),
    'steps': re.compile(r'(?:steps_applied|steps)\s*[=:]\s*(\d+)', re.I),
    'reason': re.compile(r'reason\s*[=:]\s*([a-z_]+)', re.I),
    'pytest_pass': re.compile(r'\b(PASSED|FAILED)\b.*test_grasp_smoke', re.I),
    'smoke_result': re.compile(r'Smoke result:\s*(.+)', re.I),
    'lerobot_summary': re.compile(r'(?:avg|mean|overall|total|summary).*?(?:success|rate)\s*[=:]\s*([0-9.]+)', re.I),
}

KEY_LINE_HINTS = (
    'success', 'fail', 'min_dist', 'grasp', 'raw_mag', 'action',
    'steps', 'reason', 'PASSED', 'FAILED', 'episode', 'rate',
    'Smoke result', 'PASS:', 'FAIL:',
)


def _extract(path: str | None, label: str) -> dict:
    out = {
        'label': label,
        'path': path,
        'exit_code': None,
        'hits': {},
        'key_lines': [],
        'text': '',
    }
    if not path or not os.path.exists(path):
        out['error'] = 'log not found'
        return out
    with open(path, encoding='utf-8', errors='replace') as f:
        text = f.read()
    out['text'] = text
    m = PATTERNS['exit_code'].search(text)
    if m:
        out['exit_code'] = int(m.group(1))
    for name, rx in PATTERNS.items():
        if name == 'exit_code':
            continue
        found = rx.findall(text)
        if found:
            out['hits'][name] = found[-5:]
    for line in text.splitlines():
        low = line.lower()
        if any(h.lower() in low for h in KEY_LINE_HINTS):
            out['key_lines'].append(line.strip())
    out['key_lines'] = out['key_lines'][-25:]
    return out


def _gold_success_metric(text: str) -> tuple[float | None, str]:
    if not text:
        return None, 'missing_log_text'

    candidates: list[tuple[int, float, str]] = []

    for m in re.finditer(r'success(?:_rate)?\s*[=:]\s*([0-9.]+)\s*%?', text, re.I):
        value = float(m.group(1))
        normalized = value / 100.0 if value > 1.0 else value
        candidates.append((m.start(), normalized, f'success_rate={value}'))

    for m in re.finditer(
        r'(?:success(?:_count|es)?|solved)\s*[=:]?\s*(\d+)\s*/\s*(\d+)',
        text,
        re.I,
    ):
        num = int(m.group(1))
        den = int(m.group(2))
        if den > 0:
            candidates.append((m.start(), num / den, f'success_count={num}/{den}'))

    for m in re.finditer(
        r'(?:summary|overall|final).{0,120}?(\d+)\s*/\s*(\d+).{0,80}?(?:success|solved)',
        text,
        re.I | re.S,
    ):
        num = int(m.group(1))
        den = int(m.group(2))
        if den > 0:
            candidates.append((m.start(), num / den, f'summary_success={num}/{den}'))

    if not candidates:
        return None, 'success_metric_not_found'

    candidates.sort(key=lambda x: x[0])
    _, metric, source = candidates[-1]
    return metric, source


def _print_summary(data: dict) -> None:
    print(f"\n=== {data['label']} ===")
    print(f"log: {data.get('path')}")
    if data.get('error'):
        print(data['error'])
        return
    print(f"exit_code: {data.get('exit_code')}")
    for name, vals in data.get('hits', {}).items():
        print(f"  {name}: {vals}")
    if data.get('key_lines'):
        print('  key lines:')
        for line in data['key_lines']:
            print(f'    {line}')


gold = _extract(gold_log, 'Gold (LeRobot eval)')
longor = _extract(longor_log, 'Longor grasp smoke')
_print_summary(gold)
_print_summary(longor)

gold_metric, gold_metric_src = _gold_success_metric(gold.get('text', ''))
print('\n=== Quick verdict ===')
if gold.get('exit_code') is None:
    print('Gold eval log missing or unreadable.')
elif gold.get('exit_code') != 0:
    print(f"Gold eval exited {gold['exit_code']} — investigate policy/checkpoint or eval CLI.")
elif gold_metric is None:
    print('Gold eval exited 0 but success metric was not found — result is inconclusive.')
elif gold_metric > 0:
    print(
        f'Gold eval shows non-zero success (metric={gold_metric:.3f}, source={gold_metric_src}).'
    )
else:
    print(
        f'Gold eval ran but reported zero success (metric={gold_metric:.3f}, source={gold_metric_src}).'
    )

if longor.get('exit_code') == 0:
    print('Longor grasp smoke exited 0 — integration path passed this smoke run.')
elif longor.get('exit_code') is not None:
    print(f"Longor grasp smoke exited {longor['exit_code']} — likely integration/obs/action mismatch.")
else:
    print('Longor grasp log missing or unreadable.')

if gold.get('exit_code') == 0 and gold_metric is not None and gold_metric > 0 and longor.get('exit_code') != 0:
    print('→ Gold succeeds while Longor fails: likely Longor integration gap (obs/state/action/queue wiring).')
elif gold.get('exit_code') != 0 and longor.get('exit_code') != 0:
    print('→ Both failed — checkpoint or shared sim setup may be the bottleneck.')
elif gold.get('exit_code') == 0 and gold_metric is not None and gold_metric > 0 and longor.get('exit_code') == 0:
    print('→ Both show success on this slice; compare min_dist/grasp/action lines for quality gaps.')
elif gold.get('exit_code') == 0 and gold_metric is None:
    print('→ Gold success unknown (missing metric), so cross-path conclusion is inconclusive.')

## 7 — Integration smokes

In [ ]:
# With SmolVLA server already up from section 5:
!PYTHONPATH=src MUJOCO_GL=egl PYOPENGL_PLATFORM=egl pytest -m integration tests/test_libero_integration.py -v

## 8 — Grasp debug (optional)

In [ ]:
# Optional: grasp smoke with action telemetry (server must be up from §5)
import os
# SmolVLA LIBERO trains on eef_axis_angle layout; set agent_pos to A/B test if obs has it
os.environ.setdefault('LIBERO_VLA_STATE_MODE', 'eef_axis_angle')  # eef_axis_angle | agent_pos | joints_gripper
os.environ['LIBERO_VLA_STATE_DEBUG'] = '1'  # step-0 log: eef_axis_angle vs obs agent_pos
os.environ['VLA_DEBUG_ACTION_TELEMETRY'] = '1'
!PYTHONPATH=src MUJOCO_GL=egl PYOPENGL_PLATFORM=egl pytest -m integration tests/test_libero_integration.py::test_grasp_smoke -v -s


## 9 — M7 perception wiring smoke (unit tests, no sim-GT guard)

In [ ]:
import os
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_perception_smoke_{stamp}.txt'
print(f'Capture: {capture_path}')
os.environ['M7_CAPTURE'] = capture_path
!PYTHONPATH=src pytest tests/test_libero_real_perception.py tests/test_libero_backend.py::test_libero_backend_register_skills_real_mode tests/test_libero_backend.py::test_libero_backend_register_skills tests/test_libero_backend.py::test_libero_backend_list_objects_routes_real_perception tests/test_libero_backend.py::test_libero_backend_list_objects_real_mode_no_detection_returns_empty -m 'not integration' -v 2>&1 | tee "$M7_CAPTURE"


## 10 — Run LLM example (real perception)

In [ ]:
import os, subprocess, sys
from datetime import datetime

# Set M7_STRICT_ASSERTS=1 for acceptance runs; default OFF for diagnosis loops.
strict_mode = os.environ.get('M7_STRICT_ASSERTS', '').lower() in ('1', 'true', 'yes')
print(f'Strict asserts: {"ON" if strict_mode else "OFF"} (set M7_STRICT_ASSERTS=1 for acceptance)')

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_llm_{stamp}.txt'
run_log_path = f'{trace_dir}/m7_llm_{stamp}.log'

result = subprocess.run(
    [sys.executable, 'examples/long_horizon_llm_libero.py'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=600,
    env={
        **os.environ,
        'PYTHONPATH': '/content/Longor/src',
        'MUJOCO_GL': 'egl',
        'PYOPENGL_PLATFORM': 'egl',
        'PERCEPTION_BACKEND': 'real',
        'VLA_POLICY': 'smolvla',
        'LONGOR_RUN_LOG': run_log_path,
        'LOG_LEVEL': 'DEBUG',
        'HF_HUB_DISABLE_PROGRESS_BARS': '1',
        'TQDM_DISABLE': '1',
        # Section 7 fast-fail profile: keep retries useful, avoid 10+ minute dead loops.
        'LONGOR_MAX_REPLANS': '2',
        'VLA_MAX_TOOL_SECONDS': '45',
        'VLA_MAX_INNER_STEPS': '80',
        'VLA_STAGNATION_CHUNKS': '8',
        'VLA_FAIL_WORSENING_STEPS': '40',
        'VLA_FAIL_DIST_CONSECUTIVE_STEPS': '40',
        'VLA_FAIL_IK_SATURATION_STEPS': '20',
    },
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(f'Run log: {run_log_path}')
print('=== Key lines ===')
for line in combined.splitlines():
    print(line)
print('Exit code:', result.returncode)
_log_source = combined
if os.path.exists(run_log_path):
    with open(run_log_path) as _f:
        _log_source = combined + '\n' + _f.read()

_checks = [
    (result.returncode == 0, f'exit code {result.returncode}'),
    ('ImportError' not in combined, 'Import failure detected'),
    ('ModuleNotFoundError' not in combined, 'Module not found'),
    ('Task succeeded:' in _log_source, 'E2E finished but task was not marked succeeded'),
    (
        'Run summary:' in _log_source and 'success=True' in _log_source,
        'Expected Run summary with success=True',
    ),
]
_failures = [msg for ok, msg in _checks if not ok]
if _failures:
    print('=== Acceptance checks FAILED ===')
    for msg in _failures:
        print(f'  - {msg}')
    for line in _log_source.splitlines():
        if 'Run summary:' in line or 'Task succeeded:' in line:
            print(f'  {line}')
    if strict_mode:
        raise AssertionError('; '.join(_failures))
    print('Continuing to diagnostics (strict OFF). Set M7_STRICT_ASSERTS=1 for acceptance.')
else:
    print('M7 LLM example PASS')

## 11 — Trace summary

In [ ]:
import glob, json, os

trace_dir = '/content/Longor/traces'
files = sorted(glob.glob(f'{trace_dir}/*.jsonl'))
if not files:
    print('No trace files found')
else:
    print(f'Latest trace: {files[-1]}')
    with open(files[-1]) as f:
        events = [json.loads(l) for l in f if l.strip()]
    print(f'Total events: {len(events)}')
    for e in events:
        evt = e.get('event')
        skill = e.get('skill', '')
        step = e.get('step', '')
        if evt == 'plan_start':
            skills = e.get('skills')
            if skills:
                print(f"Plan skills: {' → '.join(skills)}")
        if skill in ('arm.grab', 'arm.place'):
            r = {}
            try:
                r = json.loads(e.get('result', '{}'))
            except Exception:
                pass
            print(f"[step {step}] {skill}: status={r.get('status')} reason={r.get('reason')} steps_applied={r.get('steps_applied')} min_dist={r.get('min_dist')} last_raw={r.get('last_raw_action')}")
        elif evt == 'skill_call' and skill not in ('arm.grab', 'arm.place'):
            print(f"[step {step}] {skill}: {str(e.get('result',''))[:120]}")
        elif evt == 'feedback':
            reason = e.get('reason')
            reason_part = f' — {reason}' if reason else ''
            print(f"[step {step}] feedback {e.get('description','')!r} → {e.get('decision')}{reason_part}")
        elif evt in ('replan', 'plan_end'):
            print(json.dumps(e, indent=2))

run_logs = sorted(glob.glob(f'{trace_dir}/m7_llm_*.log'))
if run_logs:
    latest_log = run_logs[-1]
    print(f'Latest section-10 run log: {latest_log}')
    with open(latest_log) as f:
        print(f.read()[-4000:])
else:
    print('No m7_llm_*.log files found under traces')

## 12 — Pytest (non-integration)

In [ ]:
import os
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_pytest_{stamp}.txt'
print(f'Capture: {capture_path}')
os.environ['M7_CAPTURE'] = capture_path
!PYTHONPATH=src pytest -m 'not integration' --tb=short -q 2>&1 | tee "$M7_CAPTURE"


## 13 — Teardown

In [ ]:
vla_proc.terminate()
print('VLA server stopped.')